<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">La Arquitectura Encoder-Decoder</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">La combinación encoder-decoder reúne comprensión de entrada y generación de salida en un solo esquema arquitectónico. Este notebook muestra cómo encajan ambas piezas en tareas de transformación de secuencias.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Muchas aplicaciones como traducción, resumen o generación condicionada se apoyan en esta organización.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A ver el encoder y el decoder como partes complementarias de una misma arquitectura.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Una arquitectura completa no es una suma de piezas aisladas; cada bloque existe para resolver una parte concreta del flujo de información.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">El encoder construye representaciones de la entrada y el decoder las usa para producir una nueva secuencia de salida.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Definición de la secuencia de entrada
"""Este bloque define un tensor que representa una secuencia de entrada para el encoder.
Este bloque importa "torch" para trabajar con tensores en PyTorch.

Aplicamos la función "torch.randn()" a la variable "secuencia_entrada_embedded" para generar un tensor con valores aleatorios que simulan embeddings de tokens.
El primer argumento "1" indica el tamaño del batch.
El segundo argumento "6" indica la longitud de la secuencia de tokens.
El tercer argumento "16" indica la dimensión de los embeddings.

La variable "secuencia_entrada_embedded" contiene vectores densos que simulan los embeddings de los tokens de entrada.
Esta estructura permite aplicar el procesamiento del encoder sobre una secuencia numérica."""

import torch

secuencia_entrada_embedded = torch.randn(1, 6, 16)

secuencia_entrada_embedded # Muestra el tensor que representa la secuencia de entrada embebida. 


In [ ]:
# Bloque 2 — Definición de la secuencia objetivo
"""Este bloque define un tensor que representa la secuencia objetivo para el decoder.
Aplicamos la función "torch.randn()" a la variable "secuencia_objetivo_embedded" para generar un tensor con valores aleatorios que simulan los embeddings de la secuencia objetivo.
El primer argumento "1" indica el tamaño del batch.
El segundo argumento "5" indica la longitud de la secuencia de tokens.
El tercer argumento "16" indica la dimensión de los embeddings.

La variable "secuencia_objetivo_embedded" contiene vectores aleatorios que simulan embeddings de la secuencia objetivo.
Esta estructura permite aplicar atención autorregresiva dentro del decoder."""

secuencia_objetivo_embedded = torch.randn(1, 5, 16)

secuencia_objetivo_embedded # Muestra el tensor que simula los embeddings de la secuencia objetivo.


In [ ]:
# Bloque 3 — Capa de atención multicabeza del encoder
"""Este bloque define la capa de atención multicabeza utilizada en el encoder.
Este bloque importa "torch.nn" para definir capas de redes neuronales en PyTorch.

Aplicamos la clase "nn.MultiheadAttention()" a la variable "atencion_encoder" para definir una capa de atención multicabeza.
El parámetro "embed_dim=" indica la dimensionalidad de los embeddings de entrada; en este caso, 16 dimensiones.
El parámetro "num_heads=" define el número de cabezas de atención; en este caso, 2 cabezas.
El parámetro "batch_first=" indica si la primera dimensión de la entrada corresponde al tamaño del batch; en este caso, se establece como "True".
La variable "atencion_encoder" almacena una instancia de la capa configurada con los parámetros especificados.

Esta operación consiste en relacionar cada token de la secuencia con todos los demás tokens, lo que permite capturar dependencias 
a largo plazo en los datos de entrada."""

import torch.nn as nn

atencion_encoder = nn.MultiheadAttention(embed_dim=16, num_heads=2, batch_first=True)

atencion_encoder # Muestra la configuración de la capa de atención multicabeza del encoder.


In [ ]:
# Bloque 4 — Forward del encoder
"""Este bloque aplica la atención multicabeza sobre la secuencia de entrada.
Asignamos de forma múltiple la llamada a la instancia de la clase a las variables "salida_encoder_atencion" y "pesos_encoder" mediante la 
expresión "atencion_encoder(secuencia_entrada_embedded, secuencia_entrada_embedded, secuencia_entrada_embedded)".

El argumento "secuencia_entrada_embedded" es el conjunto de tensores densos de entrada que queremos procesar mediante la atención multicabeza.
Se pasa por triplicado para que la atención multicabeza pueda calcular las relaciones entre las consultas, claves y valores de la secuencia de entrada.

La variable "salida_encoder_atencion" contiene la representación contextualizada generada por la atención.
La variable "pesos_encoder" almacena los pesos de atención promediados entre cabezas por defecto."""

salida_encoder_atencion, pesos_encoder = atencion_encoder(secuencia_entrada_embedded, secuencia_entrada_embedded, secuencia_entrada_embedded)
    
salida_encoder_atencion, pesos_encoder # Muestra la salida de la atención del encoder y los pesos de atención.


In [ ]:
# Bloque 5 — Normalización y feed‑forward del encoder
"""Este bloque aplica normalización y una red feed-forward al resultado del encoder.
Aplicamos la clase "nn.LayerNorm()" a la variable "capa_norm_encoder" para definir una capa de normalización de capa.
El argumento "16" indica la dimensionalidad de los embeddings de entrada.
La variable "capa_norm_encoder" almacena una instancia de la capa de normalización de capa.

Asignamos la llamada a la instancia de la clase a la variable "salida_norm_encoder" mediante la expresión 
"capa_norm_encoder(secuencia_entrada_embedded + salida_encoder_atencion)".
El argumento "secuencia_entrada_embedded" representa la secuencia de entrada original embebida.
El argumento "salida_encoder_atencion" representa la salida de la capa de atención multicabeza del encoder.
Los valores se suman antes de aplicar la normalización de capa para mantener la información de la secuencia embebida original y la salida del encoder.

Aplicamos la clase "nn.Sequential()" a la variable "feed_forward_encoder" para definir la red feed-forward del encoder.
El primer argumento es la llamada a la clase "nn.Linear()" y define la primera capa lineal de la red feed-forward.
En este caso, la dimensionalidad de la entrada es 16 y la de la salida de la capa lineal intermedia es 32.

El segundo argumento es la llamada a la clase "nn.ReLU()" e indica la función de activación que se aplicará tras la primera capa lineal.
La función de activación introduce no linealidad en la red, lo que permite que el modelo capture relaciones más complejas entre las características de entrada.

El tercer argumento es la llamada a la clase "nn.Linear()" y define la segunda capa lineal de la red feed-forward.
En este caso, la dimensionalidad de la entrada es 32 y la de la salida de la segunda capa lineal es 16, con lo que se cierra la estructura de la red feed-forward del encoder.
La variable "feed_forward_encoder" almacena una instancia de la red feed-forward del encoder.

Asignamos la llamada a la instancia de la clase a la variable "salida_encoder" mediante la expresión "capa_norm_encoder(salida_norm_encoder + feed_forward_encoder(salida_norm_encoder))".
El argumento "salida_norm_encoder" representa la secuencia de entrada normalizada tras la combinación con la salida de atención.
El argumento "feed_forward_encoder(salida_norm_encoder)" representa la salida de la red feed-forward aplicada a la secuencia normalizada.
Ambos se suman antes de aplicar la normalización de capa para mantener la información de la secuencia original y la salida de la red feed-forward.
La variable "salida_encoder" contiene el resultado de sumar la salida feed-forward a la entrada de esa red y volver a normalizar.
El ejemplo reutiliza la misma capa de normalización en ambas conexiones residuales del encoder."""
      
capa_norm_encoder = nn.LayerNorm(16)

salida_norm_encoder = capa_norm_encoder(secuencia_entrada_embedded + salida_encoder_atencion)

feed_forward_encoder = nn.Sequential(
    nn.Linear(16, 32),
    nn.ReLU(),
    nn.Linear(32, 16)
)

salida_encoder = capa_norm_encoder(salida_norm_encoder + feed_forward_encoder(salida_norm_encoder))

salida_encoder # Muestra la salida del encoder tras la segunda conexión residual y normalización.


In [ ]:
# Bloque 6 — Atención autorregresiva del decoder
"""Este bloque define y aplica la atención autorregresiva del decoder en un solo paso.
Aplicamos la clase "nn.MultiheadAttention()" a la variable "atencion_autoregresiva" para definir una capa de atención autorregresiva.
El parámetro "embed_dim=" indica la dimensionalidad de los embeddings de entrada; en este caso, 16 dimensiones.
El parámetro "num_heads=" define el número de cabezas de atención; en este caso, 2 cabezas.
El parámetro "batch_first=" indica si la primera dimensión de la entrada corresponde al tamaño del batch; en este caso, se establece como "True".

Aplicamos el método ".size()" a la variable "secuencia_objetivo_embedded" para obtener la longitud de la secuencia objetivo.
El argumento "1" indica que queremos obtener el tamaño de la segunda dimensión de la secuencia objetivo.
La variable "longitud_objetivo" almacena la longitud de la segunda dimensión de la secuencia objetivo.

Aplicamos la función "torch.ones()" a la variable "matriz" para crear una matriz de unos de tamaño (longitud x longitud).
Los argumentos "longitud_objetivo, longitud_objetivo" indican que queremos crear una matriz de unos con el mismo número de filas y columnas que la longitud de la secuencia objetivo.
La variable "matriz" contiene una matriz de unos de "longitud_objetivo" filas y "longitud_objetivo" columnas.

Aplicamos la función "torch.tril()" acompañada del método ".unsqueeze()" a la variable "matriz" para obtener una matriz triangular 
inferior con unos en las posiciones permitidas y ceros en las futuras.
El argumento "0" del método ".unsqueeze()" indica que queremos agregar una nueva dimensión al inicio de la matriz triangular 
inferior obtenida, convirtiéndola en un tensor de tres dimensiones.

Asignamos de forma múltiple la llamada a la instancia de la clase a las variables "salida_autoregresiva" y "pesos_autoregresivos" 
mediante la expresión "atencion_autoregresiva(secuencia_objetivo_embedded, secuencia_objetivo_embedded, secuencia_objetivo_embedded,
attn_mask=(mascara_causal.squeeze(0) == 0))".

El argumento "secuencia_objetivo_embedded" es el conjunto de tensores densos de entrada que queremos procesar mediante la atención multicabeza.
Se pasa por triplicado para que la atención multicabeza pueda calcular las relaciones entre las consultas, claves y valores de la secuencia de entrada.

El parámetro "attn_mask=" recibe la máscara causal transformada para evitar que la atención considere posiciones futuras.
El método ".squeeze(0)" elimina la primera dimensión de la máscara y deja una máscara de dos dimensiones.
La comparación "== 0" convierte la máscara en booleana: "True" indica las posiciones futuras que deben bloquearse y "False", las posiciones permitidas.

La variable "salida_autoregresiva" contiene la representación contextualizada generada por la atención enmascarada.
La variable "pesos_autoregresivos" almacena los pesos de atención promediados entre cabezas por defecto."""

atencion_autoregresiva = nn.MultiheadAttention(embed_dim=16, num_heads=2, batch_first=True)

longitud_objetivo = secuencia_objetivo_embedded.size(1)

matriz = torch.ones(longitud_objetivo, longitud_objetivo)
mascara_causal = torch.tril(matriz).unsqueeze(0)

salida_autoregresiva, pesos_autoregresivos = atencion_autoregresiva(
    secuencia_objetivo_embedded,
    secuencia_objetivo_embedded,
    secuencia_objetivo_embedded,
    attn_mask=(mascara_causal.squeeze(0) == 0)
)

salida_autoregresiva, pesos_autoregresivos # Muestra la salida autorregresiva y los pesos de atención resultantes.


In [ ]:
# Bloque 7 — Atención cruzada del decoder
"""Este bloque define y aplica la atención cruzada del decoder.
Aplicamos la clase "nn.MultiheadAttention()" a la variable "atencion_cruzada" para definir la capa de atención cruzada del decoder.
El parámetro "embed_dim=" indica la dimensionalidad de los embeddings de entrada; en este caso, 16 dimensiones.
El parámetro "num_heads=" define el número de cabezas de atención; en este caso, 2 cabezas.
El parámetro "batch_first=" indica si la primera dimensión de la entrada corresponde al tamaño del batch; en este caso, se establece como "True".
La variable "atencion_cruzada" almacena una instancia de la capa que relaciona la secuencia objetivo con la salida del encoder mediante atención multicabeza.

Asignamos de forma múltiple la llamada a la instancia de la clase a las variables "salida_cruzada" y "pesos_cruzados" mediante la 
expresión "atencion_cruzada(salida_autoregresiva, salida_encoder, salida_encoder)".

El argumento "salida_autoregresiva" es la representación de la secuencia objetivo obtenida mediante atención causal.
El argumento "salida_encoder" es la representación generada por el encoder que se utilizará como claves y valores en la atención cruzada.
Este último argumento se pasa por duplicado para que la atención multicabeza pueda calcular las relaciones entre las consultas 
de la secuencia objetivo y las claves y valores de la salida del encoder.

La variable "salida_cruzada" contiene la representación generada por la atención cruzada.
La variable "pesos_cruzados" almacena los pesos de atención promediados entre cabezas por defecto."""

atencion_cruzada = nn.MultiheadAttention(embed_dim=16, num_heads=2, batch_first=True)

salida_cruzada, pesos_cruzados = atencion_cruzada(salida_autoregresiva, salida_encoder, salida_encoder)

salida_cruzada, pesos_cruzados # Muestra la salida de la atención cruzada y los pesos de atención resultantes.


In [ ]:
# Bloque 8 — Feed‑forward del decoder
"""Este bloque aplica una red feed-forward al resultado del decoder.
Aplicamos la clase "nn.Sequential()" a la variable "feed_forward_decoder" para definir la red feed-forward del decoder.

El primer argumento es la llamada a la clase "nn.Linear()" y define la primera capa lineal de la red feed-forward.
En este caso, la dimensionalidad de la entrada es 16 y la de la salida de la capa lineal intermedia es 32.

El segundo argumento es la llamada a la clase "nn.ReLU()" e indica la función de activación que se aplicará tras la primera capa lineal.
La función de activación introduce no linealidad en la red, lo que permite que el modelo capture relaciones más complejas entre las características de entrada.

El tercer argumento es la llamada a la clase "nn.Linear()" y define la segunda capa lineal de la red feed-forward.
En este caso, la dimensionalidad de la entrada es 32 y la de la salida de la segunda capa lineal es 16, con lo que se cierra la estructura de la red feed-forward del decoder.
La variable "feed_forward_decoder" almacena una instancia de la red feed-forward del decoder.

Asignamos la llamada a la instancia de la clase a la variable "salida_decoder" mediante la expresión "feed_forward_decoder(salida_cruzada)".
El argumento "salida_cruzada" representa la salida de la atención cruzada, sin normalización ni conexión residual adicional en el decoder.
La variable "salida_decoder" contiene la salida de la red feed-forward aplicada a esa representación.
Este decoder simplificado no incluye conexiones residuales, normalización, codificación posicional ni generación de tokens."""

feed_forward_decoder = nn.Sequential(
    nn.Linear(16, 32),
    nn.ReLU(),
    nn.Linear(32, 16)
)

salida_decoder = feed_forward_decoder(salida_cruzada)

salida_decoder # Muestra la salida de la red feed-forward del decoder.


In [ ]:
# Bloque 9 — Resultados finales
"""Este bloque organiza los resultados del procesamiento encoder-decoder.
La variable "resultado" almacena en forma de diccionario las salidas del encoder, del decoder y los pesos de atención de cada componente.
La estructura permite visualizar cada componente de la arquitectura encoder-decoder."""

resultado = {
    "salida_encoder": salida_encoder,
    "pesos_encoder": pesos_encoder,
    "salida_autoregresiva": salida_autoregresiva,
    "pesos_autoregresivos": pesos_autoregresivos,
    "salida_cruzada": salida_cruzada,
    "pesos_cruzados": pesos_cruzados,
    "salida_decoder": salida_decoder
}

resultado # Muestra el diccionario con los resultados finales del procesamiento encoder-decoder.

<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto cómo se articula una arquitectura completa capaz de transformar una secuencia de entrada en otra de salida.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">Encoder y decoder cooperan, pero cumplen funciones distintas dentro del mismo sistema.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Estudiar cada bloque por separado y perder la lógica de intercambio de información entre ambos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en hugging Face para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en hugging Face y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">05_Hugging_face/01_Hugging_face.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">